# MoodFit V4 - Street cap plus more trainable layers
V3와 동일하게 스트리트 학습 이미지를 10,000장으로 제한하고, ConvNeXt 마지막 3개 모듈을 미세조정합니다.

In [ ]:
# V4 이어 학습: 기존 최고 모델에서 마지막 기록 다음 에포크부터 시작
# 주의: 위의 기존 V4 전체 학습 셀은 실행하지 말고, 준비 셀 실행 후 이 셀만 실행하세요.
import re

model_dir = Path("Model")
model_dir.mkdir(exist_ok=True)
street_cap = 10000
log_path = model_dir / f"v4_street_cap_{street_cap}_more_layers_training_log.txt"
best_path = model_dir / f"fashion_mood_model_pytorch4_street_cap_{street_cap}_more_layers_best.pth"
last_path = model_dir / f"fashion_mood_model_pytorch4_street_cap_{street_cap}_more_layers_last.pth"

resume_batch_size = 16
target_total_epochs = 30  # 더 진행하려면 50 또는 100으로 변경
resume_loader_kwargs = dict(num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
train_loader = DataLoader(train_dataset, batch_size=resume_batch_size, shuffle=True, **resume_loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=resume_batch_size, shuffle=False, **resume_loader_kwargs)

if not best_path.exists() and not last_path.exists():
    raise FileNotFoundError(f"V4 체크포인트를 찾을 수 없습니다: {best_path}")
resume_path = last_path if last_path.exists() else best_path
mood_model.load_state_dict(torch.load(resume_path, map_location="cpu", weights_only=True))

feature_modules = list(mood_model.features.children())
for param in mood_model.parameters():
    param.requires_grad = False
for module in feature_modules[-3:]:
    for param in module.parameters():
        param.requires_grad = True
mood_model = mood_model.to(device)
backbone_params = [param for param in mood_model.features.parameters() if param.requires_grad]
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": 3e-6},
    {"params": mood_model.classifier.parameters(), "lr": 1e-4},
], weight_decay=1e-4)
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)

record_pattern = re.compile(r"^Epoch \[(\d+)(?:/\d+)?\].*Val Macro F1: ([0-9.]+)")
records = []
if log_path.exists():
    for line in log_path.read_text(encoding="utf-8").splitlines():
        match = record_pattern.match(line)
        if match:
            records.append((int(match.group(1)), float(match.group(2))))
start_epoch = max((epoch for epoch, _ in records), default=0)
best_epoch = max(records, key=lambda item: item[1])[0] if records else 0
best_macro_f1 = max((score for _, score in records), default=-1.0)
print(f"{resume_path}에서 불러옴 | {start_epoch}에포크 다음부터 | batch_size={resume_batch_size}")

for epoch in range(start_epoch, target_total_epochs):
    actual_epoch = epoch + 1
    mood_model.train()
    for module in feature_modules[:-3]:
        module.eval()
    for module in feature_modules[-3:]:
        module.train()
    mood_model.classifier.train()
    running_loss = 0.0
    correct_train = total_train = 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {actual_epoch}/{target_total_epochs} [Train]", unit="batch"):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            outputs = mood_model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)
        total_train += labels.size(0)
        correct_train += (outputs.argmax(1) == labels).sum().item()
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train

    mood_model.eval()
    val_running_loss = 0.0
    correct_val = total_val = 0
    actual_labels, predicted_labels = [], []
    with torch.inference_mode():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {actual_epoch}/{target_total_epochs} [Valid]", unit="batch"):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                outputs = mood_model(inputs)
                loss = criterion(outputs, labels)
            predicted = outputs.argmax(1)
            val_running_loss += loss.item() * labels.size(0)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            actual_labels.extend(labels.cpu().tolist())
            predicted_labels.extend(predicted.cpu().tolist())
    val_loss = val_running_loss / total_val
    val_acc = correct_val / total_val
    val_macro_f1 = f1_score(actual_labels, predicted_labels, labels=list(range(num_classes)), average="macro", zero_division=0)
    val_balanced_acc = balanced_accuracy_score(actual_labels, predicted_labels)
    log_line = (f"Epoch [{actual_epoch}/{target_total_epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
                 f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%} | Val Macro F1: {val_macro_f1:.4f} | "
                 f"Val Balanced Acc: {val_balanced_acc:.2%}")
    print(log_line, flush=True)
    with log_path.open("a", encoding="utf-8") as log_file:
        log_file.write(log_line + "\n")
    torch.save(mood_model.state_dict(), last_path)
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = actual_epoch
        torch.save(mood_model.state_dict(), best_path)
        print(f"최고 모델 저장: {best_path}")

print(f"V4 이어 학습 완료 | best epoch={best_epoch} | best Macro F1={best_macro_f1:.4f}")

Model\fashion_mood_model_pytorch4_street_cap_10000_more_layers_best.pth에서 불러옴 | 4에포크 다음부터 | batch_size=16


Epoch 5/30 [Train]:   0%|          | 0/3867 [00:16<?, ?batch/s]

Epoch 5/30 [Valid]:   0%|          | 0/1513 [00:16<?, ?batch/s]

Epoch [5/30] | Train Loss: 1.8351 | Train Acc: 39.99% | Val Loss: 1.6651 | Val Acc: 47.38% | Val Macro F1: 0.1819 | Val Balanced Acc: 18.65%
최고 모델 저장: Model\fashion_mood_model_pytorch4_street_cap_10000_more_layers_best.pth


Epoch 6/30 [Train]:   0%|          | 0/3867 [00:00<?, ?batch/s]

Epoch 6/30 [Valid]:   0%|          | 0/1513 [00:00<?, ?batch/s]

... (17 outputs left out)

Epoch 12/30 [Valid]:   0%|          | 0/1513 [00:00<?, ?batch/s]

Epoch [12/30] | Train Loss: 1.6528 | Train Acc: 45.34% | Val Loss: 1.6053 | Val Acc: 48.36% | Val Macro F1: 0.2063 | Val Balanced Acc: 21.22%
최고 모델 저장: Model\fashion_mood_model_pytorch4_street_cap_10000_more_layers_best.pth


Epoch 13/30 [Train]:   0%|          | 0/3867 [00:00<?, ?batch/s]

KeyboardInterrupt: 

In [ ]:
import copy
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU를 사용할 수 없습니다.")
device = torch.device("cuda:0")
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
print(f"사용 디바이스: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
dataset_path = "CNN_train_test/CNN_train_test_image"
street_class = "스트리트"
street_cap = 10_000
batch_size = 32
workers = 4
eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05), interpolation=transforms.InterpolationMode.BILINEAR, fill=(128, 128, 128)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
full_dataset = datasets.ImageFolder(dataset_path, transform=eval_transforms)
class_names = full_dataset.classes
num_classes = len(class_names)
targets = np.asarray(full_dataset.targets)
street_label = class_names.index(street_class)
train_ids, val_ids = train_test_split(np.arange(len(full_dataset)), test_size=0.2, stratify=targets, random_state=SEED)
rng = np.random.default_rng(SEED)
street_train_ids = train_ids[targets[train_ids] == street_label]
other_train_ids = train_ids[targets[train_ids] != street_label]
if len(street_train_ids) > street_cap:
    street_train_ids = rng.choice(street_train_ids, street_cap, replace=False)
capped_train_ids = np.concatenate([other_train_ids, street_train_ids])
rng.shuffle(capped_train_ids)
train_source = copy.copy(full_dataset)
train_source.transform = train_transforms
train_dataset = Subset(train_source, capped_train_ids.tolist())
val_dataset = Subset(full_dataset, val_ids.tolist())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True, persistent_workers=workers > 0)
train_counts = np.bincount(targets[capped_train_ids], minlength=num_classes)
print(f"전체 이미지: {len(full_dataset):,}장")
print(f"V4 학습 이미지: {len(train_dataset):,}장")
print(f"V4 검증 이미지: {len(val_dataset):,}장")
print(f"스트리트 학습 이미지: {train_counts[street_label]:,}장")
print(f"학습 배치: {len(train_loader):,}개/epoch")
display(pd.DataFrame({"class": class_names, "train_images": train_counts}))

In [ ]:
weights = models.ConvNeXt_Tiny_Weights.DEFAULT
mood_model = models.convnext_tiny(weights=weights)
for param in mood_model.parameters():
    param.requires_grad = False
in_features = mood_model.classifier[2].in_features
mood_model.classifier = nn.Sequential(nn.Flatten(1), nn.Linear(in_features, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
feature_modules = list(mood_model.features.children())
unfrozen_modules = feature_modules[-3:]
for module in unfrozen_modules:
    for param in module.parameters():
        param.requires_grad = True
mood_model = mood_model.to(device)
criterion = nn.CrossEntropyLoss()
backbone_params = [param for param in mood_model.features.parameters() if param.requires_grad]
optimizer = optim.AdamW([{"params": backbone_params, "lr": 3e-6}, {"params": mood_model.classifier.parameters(), "lr": 1e-4}], weight_decay=1e-4)
amp_enabled = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
trainable_params = sum(param.numel() for param in mood_model.parameters() if param.requires_grad)
print(f"학습 가능한 파라미터: {trainable_params:,}")
print("마지막 3개 ConvNeXt 모듈을 학습합니다.")

In [ ]:
epochs = 100
train_loss_history, val_loss_history = [], []
train_acc_history, val_acc_history = [], []
val_macro_f1_history, val_balanced_acc_history = [], []
model_dir = Path("Model")
model_dir.mkdir(exist_ok=True)
log_path = model_dir / f"v4_street_cap_{street_cap}_more_layers_training_log.txt"
best_path = model_dir / f"fashion_mood_model_pytorch4_street_cap_{street_cap}_more_layers_best.pth"
last_path = model_dir / f"fashion_mood_model_pytorch4_street_cap_{street_cap}_more_layers_last.pth"
metadata_path = model_dir / f"fashion_mood_model_pytorch4_street_cap_{street_cap}_more_layers.json"
with log_path.open("a", encoding="utf-8") as log_file:
    log_file.write("\n\n" + "=" * 100 + "\n")
    log_file.write(f"V4: street cap={street_cap:,}, last 3 feature modules trainable\n")
    log_file.write("=" * 100 + "\n")
best_macro_f1 = -1.0
best_epoch = 0
for epoch in range(epochs):
    mood_model.train()
    for module in feature_modules[:-3]:
        module.eval()
    for module in feature_modules[-3:]:
        module.train()
    mood_model.classifier.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Train]", unit="batch"):
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
            outputs = mood_model(inputs)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * labels.size(0)
        total_train += labels.size(0)
        correct_train += (outputs.argmax(1) == labels).sum().item()
    train_loss = running_loss / total_train
    train_acc = correct_train / total_train
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)
    mood_model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    actual_labels, predicted_labels = [], []
    with torch.inference_mode():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{epochs} [Valid]", unit="batch"):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                outputs = mood_model(inputs)
                loss = criterion(outputs, labels)
            predicted = outputs.argmax(1)
            val_running_loss += loss.item() * labels.size(0)
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()
            actual_labels.extend(labels.cpu().tolist())
            predicted_labels.extend(predicted.cpu().tolist())
    val_loss = val_running_loss / total_val
    val_acc = correct_val / total_val
    val_macro_f1 = f1_score(actual_labels, predicted_labels, labels=list(range(num_classes)), average="macro", zero_division=0)
    val_balanced_acc = balanced_accuracy_score(actual_labels, predicted_labels)
    val_loss_history.append(val_loss)
    val_acc_history.append(val_acc)
    val_macro_f1_history.append(val_macro_f1)
    val_balanced_acc_history.append(val_balanced_acc)
    log_line = f"Epoch [{epoch + 1}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2%} | Val Macro F1: {val_macro_f1:.4f} | Val Balanced Acc: {val_balanced_acc:.2%}"
    print(log_line, flush=True)
    with log_path.open("a", encoding="utf-8") as log_file:
        log_file.write(log_line + "\n")
    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = epoch + 1
        torch.save(mood_model.state_dict(), best_path)
        print(f"최고 모델 저장: {best_path}")
torch.save(mood_model.state_dict(), last_path)
metadata_path.write_text(json.dumps({"class_names": class_names, "street_class": street_class, "street_cap": street_cap, "unfrozen_feature_modules": 3, "best_epoch": best_epoch, "best_val_macro_f1": best_macro_f1}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"V4 학습 완료: best epoch={best_epoch}, best Macro F1={best_macro_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# 학습 셀을 건너뛰어도 저장된 로그에서 그래프를 복원한다.
plot_log_path = Path("Model/v4_street_cap_10000_more_layers_training_log.txt")
train_loss_history, train_acc_history = [], []
val_loss_history, val_acc_history = [], []
val_macro_f1_history, val_balanced_acc_history = [], []
for line in plot_log_path.read_text(encoding="utf-8").splitlines():
    if not line.startswith("Epoch ["):
        continue
    parts = [part.strip() for part in line.split("|")]
    if len(parts) != 7:
        continue
    train_loss_history.append(float(parts[1].split(": ", 1)[1]))
    train_acc_history.append(float(parts[2].split(": ", 1)[1].rstrip("%")) / 100)
    val_loss_history.append(float(parts[3].split(": ", 1)[1]))
    val_acc_history.append(float(parts[4].split(": ", 1)[1].rstrip("%")) / 100)
    val_macro_f1_history.append(float(parts[5].split(": ", 1)[1]))
    val_balanced_acc_history.append(float(parts[6].split(": ", 1)[1].rstrip("%")) / 100)
if not train_loss_history:
    raise RuntimeError(f"로그에서 에포크 기록을 찾지 못했습니다: {plot_log_path}")
x = range(1, len(train_loss_history) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(x, train_loss_history, label="Train Loss")
axes[0].plot(x, val_loss_history, label="Val Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[1].plot(x, train_acc_history, label="Train Acc")
axes[1].plot(x, val_acc_history, label="Val Acc")
axes[1].set_title("Accuracy")
axes[1].legend()
axes[2].plot(x, val_macro_f1_history, label="Val Macro F1")
axes[2].plot(x, val_balanced_acc_history, label="Val Balanced Acc")
axes[2].set_title("Balanced metrics")
axes[2].legend()
for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout()
plt.show()
best_idx = int(np.argmax(val_macro_f1_history))
print(f"최고 Macro F1 epoch: {best_idx + 1}")
print(f"Val Macro F1: {val_macro_f1_history[best_idx]:.4f}")
print(f"Val Acc: {val_acc_history[best_idx]:.2%}")
print(f"Val Balanced Acc: {val_balanced_acc_history[best_idx]:.2%}")